# 23. π0 and FAST — VLA flow policy and action tokenizer

π0 keeps its released structural path at reduced tensor width. FAST keeps the actual DCT/quantization/frequency-major/BPE round trip.


In [ ]:
import math
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(12)
device = torch.device("cpu")


## 1. π0 — 27-layer SigLIP tower, 18 joint layers, GQA, RoPE, block mask, Beta-time flow matching


In [ ]:
class VisionBlock(nn.Module):
    def __init__(self, dim=32, heads=16):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.GELU(),
            nn.Linear(4 * dim, dim),
        )

    def forward(self, x):
        normalized = self.norm1(x)
        attended, _ = self.attn(
            normalized,
            normalized,
            normalized,
            need_weights=False,
        )
        x = x + attended
        return x + self.mlp(self.norm2(x))


class SigLIPVision(nn.Module):
    def __init__(self, dim=32, image_size=28, patch=14, depth=27):
        super().__init__()
        self.patch = nn.Conv2d(3, dim, patch, stride=patch)
        patch_count = (image_size // patch) ** 2
        self.position = nn.Parameter(
            torch.randn(1, patch_count, dim) * 0.02
        )
        self.blocks = nn.ModuleList(
            [VisionBlock(dim, 16) for _ in range(depth)]
        )
        self.norm = nn.LayerNorm(dim)

    def forward(self, image):
        hidden = self.patch(image).flatten(2).transpose(1, 2)
        hidden = hidden + self.position[:, : hidden.size(1)]
        for block in self.blocks:
            hidden = block(hidden)
        return self.norm(hidden)


def block_attention_mask(valid, ar_mask):
    ar_mask = ar_mask[None].expand_as(valid)
    groups = torch.cumsum(ar_mask.long(), dim=1)
    causal = groups[:, None, :] <= groups[:, :, None]
    return causal & valid[:, None, :] & valid[:, :, None]


def pi0_time_embedding(t, dim, min_period=4e-3, max_period=4.0):
    half = dim // 2
    fraction = torch.linspace(0, 1, half, device=t.device)
    period = min_period * (max_period / min_period) ** fraction
    angle = t[:, None] * (2 * math.pi / period)[None]
    return torch.cat([angle.sin(), angle.cos()], dim=-1)


def apply_rope(x, position):
    dim = x.size(-1)
    pair_index = torch.arange(
        0,
        dim,
        2,
        device=x.device,
        dtype=torch.float32,
    )
    inverse_frequency = 1.0 / (10000 ** (pair_index / dim))
    angle = (
        position[:, None, :, None].float()
        * inverse_frequency[None, None, None]
    )
    even = x[..., 0::2]
    odd = x[..., 1::2]
    rotated_even = even * angle.cos() - odd * angle.sin()
    rotated_odd = even * angle.sin() + odd * angle.cos()
    return torch.stack(
        [rotated_even, rotated_odd],
        dim=-1,
    ).flatten(-2)


class GemmaFFN(nn.Module):
    def __init__(self, dim=32, hidden_dim=128):
        super().__init__()
        self.gate = nn.Linear(dim, hidden_dim, bias=False)
        self.value = nn.Linear(dim, hidden_dim, bias=False)
        self.output = nn.Linear(hidden_dim, dim, bias=False)

    def forward(self, x):
        gated = F.gelu(self.gate(x), approximate="tanh")
        value = self.value(x)
        return self.output(gated * value)


class Pi0Layer(nn.Module):
    def __init__(self, dim=32, q_heads=8, kv_heads=1):
        super().__init__()
        assert q_heads % kv_heads == 0
        self.q_heads = q_heads
        self.kv_heads = kv_heads
        self.head_dim = dim // q_heads
        self.repeat = q_heads // kv_heads

        self.prefix_attn_norm = nn.RMSNorm(dim)
        self.suffix_attn_norm = nn.RMSNorm(dim)
        self.prefix_q = nn.Linear(
            dim,
            q_heads * self.head_dim,
            bias=False,
        )
        self.prefix_kv = nn.Linear(
            dim,
            2 * kv_heads * self.head_dim,
            bias=False,
        )
        self.suffix_q = nn.Linear(
            dim,
            q_heads * self.head_dim,
            bias=False,
        )
        self.suffix_kv = nn.Linear(
            dim,
            2 * kv_heads * self.head_dim,
            bias=False,
        )
        self.prefix_out = nn.Linear(dim, dim, bias=False)
        self.suffix_out = nn.Linear(dim, dim, bias=False)

        self.prefix_ffn_norm = nn.RMSNorm(dim)
        self.suffix_ffn_norm = nn.RMSNorm(dim)
        self.prefix_ffn = GemmaFFN(dim, 4 * dim)
        self.suffix_ffn = GemmaFFN(dim, 4 * dim)

    def project(self, x, q_projection, kv_projection, position):
        batch, length, _ = x.shape
        q = q_projection(x).view(
            batch,
            length,
            self.q_heads,
            self.head_dim,
        ).transpose(1, 2)
        kv = kv_projection(x).view(
            batch,
            length,
            2,
            self.kv_heads,
            self.head_dim,
        )
        k, v = kv.permute(2, 0, 3, 1, 4).unbind(0)
        q = apply_rope(q, position)
        k = apply_rope(k, position)
        k = k.repeat_interleave(self.repeat, dim=1)
        v = v.repeat_interleave(self.repeat, dim=1)
        return q, k, v

    def forward(self, prefix, suffix, mask, position):
        split = prefix.size(1)
        prefix_normalized = self.prefix_attn_norm(prefix)
        suffix_normalized = self.suffix_attn_norm(suffix)

        prefix_q, prefix_k, prefix_v = self.project(
            prefix_normalized,
            self.prefix_q,
            self.prefix_kv,
            position[:, :split],
        )
        suffix_q, suffix_k, suffix_v = self.project(
            suffix_normalized,
            self.suffix_q,
            self.suffix_kv,
            position[:, split:],
        )

        q = torch.cat([prefix_q, suffix_q], dim=2)
        k = torch.cat([prefix_k, suffix_k], dim=2)
        v = torch.cat([prefix_v, suffix_v], dim=2)
        q = q * (self.head_dim ** -0.5)

        additive_mask = torch.zeros_like(mask, dtype=q.dtype)
        additive_mask = additive_mask.masked_fill(
            ~mask,
            torch.finfo(q.dtype).min,
        )
        attended = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=additive_mask[:, None],
            scale=1.0,
        )
        attended = attended.transpose(1, 2).contiguous().flatten(2)

        prefix = prefix + self.prefix_out(attended[:, :split])
        suffix = suffix + self.suffix_out(attended[:, split:])
        prefix = prefix + self.prefix_ffn(
            self.prefix_ffn_norm(prefix)
        )
        suffix = suffix + self.suffix_ffn(
            self.suffix_ffn_norm(suffix)
        )
        return prefix, suffix


class Pi0(nn.Module):
    def __init__(self, dim=32, action_dim=3, horizon=4, vocab=64):
        super().__init__()
        self.horizon = horizon
        self.action_dim = action_dim
        self.dim = dim
        self.vision = SigLIPVision(dim)
        self.language = nn.Embedding(vocab, dim)
        self.state = nn.Linear(action_dim, dim)
        self.action = nn.Linear(action_dim, dim)
        self.action_time = nn.Sequential(
            nn.Linear(2 * dim, dim),
            nn.SiLU(),
            nn.Linear(dim, dim),
        )
        self.layers = nn.ModuleList(
            [Pi0Layer(dim) for _ in range(18)]
        )
        self.prefix_final_norm = nn.RMSNorm(dim)
        self.suffix_final_norm = nn.RMSNorm(dim)
        self.out = nn.Linear(dim, action_dim)

    def forward(self, image, language, state, noisy_action, t):
        vision_tokens = self.vision(image)
        language_tokens = self.language(language) * math.sqrt(self.dim)
        prefix = torch.cat([vision_tokens, language_tokens], dim=1)

        state_token = self.state(state).unsqueeze(1)
        time = pi0_time_embedding(t, self.dim)
        time = time[:, None].expand(-1, self.horizon, -1)
        action_tokens = self.action(noisy_action)
        action_tokens = self.action_time(
            torch.cat([action_tokens, time], dim=-1)
        )
        suffix = torch.cat([state_token, action_tokens], dim=1)

        total_length = prefix.size(1) + suffix.size(1)
        valid = torch.ones(
            prefix.size(0),
            total_length,
            dtype=torch.bool,
            device=prefix.device,
        )
        prefix_ar = torch.zeros(
            prefix.size(1),
            dtype=torch.bool,
            device=prefix.device,
        )
        suffix_ar = torch.tensor(
            [True, True] + [False] * (self.horizon - 1),
            dtype=torch.bool,
            device=prefix.device,
        )
        mask = block_attention_mask(
            valid,
            torch.cat([prefix_ar, suffix_ar]),
        )
        position = torch.cumsum(valid.long(), dim=1) - 1

        for layer in self.layers:
            prefix, suffix = layer(
                prefix,
                suffix,
                mask,
                position,
            )

        prefix = self.prefix_final_norm(prefix)
        suffix = self.suffix_final_norm(suffix)
        del prefix
        return self.out(suffix[:, -self.horizon:])


@torch.no_grad()
def sample_pi0(model, image, language, state, steps=10):
    action = torch.randn(
        image.size(0),
        model.horizon,
        model.action_dim,
        device=image.device,
    )
    dt = -1.0 / steps

    for step in range(steps):
        t_value = 1.0 - step / steps
        t = torch.full(
            (image.size(0),),
            t_value,
            device=image.device,
        )
        velocity = model(
            image,
            language,
            state,
            action,
            t,
        )
        action = action + dt * velocity
    return action


pi0 = Pi0()
assert len(pi0.vision.blocks) == 27
assert len(pi0.layers) == 18
assert pi0.layers[0].q_heads == 8
assert pi0.layers[0].kv_heads == 1

image = torch.randn(1, 3, 28, 28)
language = torch.randint(0, 64, (1, 3))
state = torch.randn(1, 3)
actions = torch.randn(1, 4, 3)
noise = torch.randn_like(actions)
t = torch.distributions.Beta(1.5, 1.0).sample((1,))
t = t * 0.999 + 0.001
x_t = t[:, None, None] * noise + (1 - t[:, None, None]) * actions
target_velocity = noise - actions
predicted_velocity = pi0(
    image,
    language,
    state,
    x_t,
    t,
)
F.mse_loss(predicted_velocity, target_velocity).backward()

sampled_actions = sample_pi0(
    pi0,
    image,
    language,
    state,
    steps=10,
)
assert sampled_actions.shape == actions.shape


## 2. FAST — normalization, DCT, quantization, frequency-major flattening, BPE round trip


In [ ]:
def dct_matrix(length, device):
    time_index = torch.arange(
        length,
        device=device,
        dtype=torch.float32,
    )
    frequency_index = torch.arange(
        length,
        device=device,
        dtype=torch.float32,
    )[:, None]
    matrix = torch.cos(
        math.pi
        / length
        * (time_index + 0.5)
        * frequency_index
    )
    matrix[0] *= math.sqrt(1 / length)
    matrix[1:] *= math.sqrt(2 / length)
    return matrix


def normalize_actions(actions, low, high):
    return (
        2
        * (actions - low)
        / (high - low).clamp_min(1e-6)
        - 1
    )


def denormalize_actions(actions, low, high):
    return 0.5 * (actions + 1) * (high - low) + low


def quantize(coefficients, scale=64.0):
    return torch.round(coefficients * scale).long()


def dequantize(tokens, scale=64.0):
    return tokens.float() / scale


def train_bpe(symbols, merges=4):
    sequence = list(symbols)
    merge_rules = []

    for _ in range(merges):
        pairs = Counter(zip(sequence[:-1], sequence[1:]))
        if not pairs:
            break
        pair, _ = pairs.most_common(1)[0]
        merged_symbol = (pair[0], pair[1])
        new_sequence = []
        index = 0

        while index < len(sequence):
            has_pair = (
                index + 1 < len(sequence)
                and (sequence[index], sequence[index + 1]) == pair
            )
            if has_pair:
                new_sequence.append(merged_symbol)
                index += 2
            else:
                new_sequence.append(sequence[index])
                index += 1

        merge_rules.append(pair)
        sequence = new_sequence

    return sequence, merge_rules


def expand_bpe(tokens):
    output = []
    for token in tokens:
        if isinstance(token, tuple):
            output.extend(expand_bpe(list(token)))
        else:
            output.append(token)
    return output


def fast_encode(actions, merges=4, quantization_scale=64.0):
    low = torch.quantile(
        actions,
        0.01,
        dim=1,
        keepdim=True,
    )
    high = torch.quantile(
        actions,
        0.99,
        dim=1,
        keepdim=True,
    )
    normalized = normalize_actions(actions, low, high)
    matrix = dct_matrix(actions.size(1), actions.device)
    coefficients = torch.einsum(
        "ft,btd->bfd",
        matrix,
        normalized,
    )
    quantized = quantize(coefficients, quantization_scale)
    frequency_major = quantized.permute(0, 2, 1).reshape(-1)
    symbols = frequency_major.tolist()
    bpe_tokens, rules = train_bpe(symbols, merges)
    metadata = {
        "shape": quantized.shape,
        "low": low,
        "high": high,
        "matrix": matrix,
        "scale": quantization_scale,
        "symbols": symbols,
        "rules": rules,
    }
    return bpe_tokens, metadata


def fast_decode(bpe_tokens, metadata):
    symbols = expand_bpe(bpe_tokens)
    quantized = torch.tensor(
        symbols,
        device=metadata["matrix"].device,
        dtype=torch.long,
    )
    batch, frequencies, dimensions = metadata["shape"]
    quantized = quantized.view(
        batch,
        dimensions,
        frequencies,
    ).permute(0, 2, 1)
    coefficients = dequantize(
        quantized,
        metadata["scale"],
    )
    normalized = torch.einsum(
        "tf,bfd->btd",
        metadata["matrix"].T,
        coefficients,
    )
    return denormalize_actions(
        normalized,
        metadata["low"],
        metadata["high"],
    )


actions = torch.randn(1, 8, 3)
bpe_tokens, metadata = fast_encode(actions)
reconstructed = fast_decode(bpe_tokens, metadata)
assert expand_bpe(bpe_tokens) == metadata["symbols"]
assert reconstructed.shape == actions.shape
assert torch.isfinite(reconstructed).all()


## Audit result

π0 executes the 27-layer SigLIP tower, 18-layer base/action-expert joint stack, GQA, RoPE, Gemma-style gated FFNs, prefix/state/action block attention, Beta(1.5, 1.0) flow-matching training, final expert normalization, and Euler rollout. FAST performs quantile normalization, DCT, quantization, frequency-major flattening, actual BPE merge expansion, dequantization, inverse DCT, and denormalization.
